# University of London - ML Code - Computer Science Final Project

**BSc Computer Science**

**Subject: CM3070 Computer Science Final Project**

**Student: In Final Project report**

**Student Number: In Final Project report**

## ML Project Iteration Two

Based on the fact that the models trained with KerasTuner using the Random Search algorithm proved to have the best performance in comparison to the baseline, the 1D convolutional model and the ones trained using the Genetic Algorithm, I decided to do another iteration in which, as initially proposed, the models will be trained not only with the historical price of the currency, but also with other economic data. In this case, I decided to use the inflation rate of each country.

The result of this notebook are 3 models per currency pair (each with a delay longer than the previous one), using the KerasTuner procedure, used in the ml_process notebook.

## Preparing the data for KerasTuner

### Currencies

In [1]:
currencies = [
    "jpy",
    "aud",
    "cad",
    "cny",
    "hkd",
    "chf",
]

### Transforming the csv data to a numpy array

In [2]:
import numpy as np

str_to_np_date = lambda x: np.datetime64(x)

float_to_two_decimal_float = lambda x: round(number=float(x), ndigits=2)

currencies_raw_data = {}

# Pairs of currencies
for currency in currencies:
    raw_data = np.genfromtxt(f"./data/currency-data/USD-{currency.upper()}-DAILY-WITH-INFLATION-RATE.csv", skip_header=1, delimiter=";", usecols=(1,2), converters={1: float_to_two_decimal_float, 2: float_to_two_decimal_float})
    raw_data_dates = np.genfromtxt(f"./data/currency-data/USD-{currency.upper()}-DAILY-WITH-INFLATION-RATE.csv", skip_header=1, delimiter=";", usecols=0, converters={0: str_to_np_date})
    raw_data_for_evals = np.genfromtxt(f"./data/currency-data/USD-{currency.upper()}-DAILY.csv", skip_header=1, delimiter=";", usecols=1)

    # As the currency data is from newer to older, the order should be inverted.
    raw_data = np.flip(raw_data, axis=0)
    raw_data_dates = np.flip(raw_data_dates, axis=0)
    raw_data_for_evals = np.flip(raw_data_for_evals, axis=0)

    # calculate train sample number
    train_samples_number = len(raw_data)

    currencies_raw_data[f"{currency}"] = {
        f"usd_{currency}_train_samples_number": train_samples_number,
        f"usd_{currency}_raw_data": raw_data,
        f"usd_{currency}_raw_data_dates": raw_data_dates,
        f"usd_{currency}_raw_data_for_evals": raw_data_for_evals
    }

    # Print essential data information
    # print(f"Length usd_{currency}_raw_data: ",len(raw_data))
    # print(f"Data type usd_{currency}_raw_data: ",raw_data.dtype)
    # print(f"Raw Data Sample - usd_{currency}_raw_data: ",raw_data[:10])
    # print(f"Raw Data Dates Sample - usd_{currency}_raw_data_dates: ",raw_data_dates[:10])
    # print(f"Number of train samples - usd_{currency}_raw_data_dates: ", train_samples_number)

# Check one of the currencies in the dictionary
# print(currencies_raw_data["jpy"][f"usd_jpy_raw_data"][:10])
# print(currencies_raw_data["jpy"][f"usd_jpy_raw_data_dates"][:10])

# print(currencies_raw_data)

### Creating timeseries data

Based on the code provided in Chapter 10 (Deep Learning For Timeseries) of the book Deep Learning with Python by Francois Chollet [1].

In [18]:
import tensorflow as tf

# Parameters: sampling_rate, sequence_length, delay, and batch_size
sampling_rate = 1
sequence_length = 150 # Observations will go back 150 days
delays = []

for i in range(3):
    delays.append(sampling_rate * (sequence_length + (5 + (20 * i)) - 1)) # target is 5, 25, 45 days after the end of the sequence

train_datasets = {}
train_datasets_for_eval_prev_models = {}

data_inputs_test_all_for_eval_prev_models = {}
data_outputs_test_all_for_eval_prev_models = {}
data_outputs_test_all = {}
data_inputs_test_all = {}
data_outputs_test_all = {}
data_inputs_all = {}
data_outputs_all = {}

for index, delay in enumerate(delays):
    for currency in currencies:

        # normalize data
        # Based on the code provided in Chapter 10 (Deep Learning For Timeseries) of the book Deep Learning with Python by Francois Chollet [1].
        currency_data_temp = currencies_raw_data[currency][f"usd_{currency}_raw_data"]
        mean = currency_data_temp[:currencies_raw_data[currency][f"usd_{currency}_train_samples_number"]].mean(axis=0)
        currency_data_temp -= mean
        std = currency_data_temp[:currencies_raw_data[currency][f"usd_{currency}_train_samples_number"]].std(axis=0)
        currency_data_temp /= std

        # train dataset (with normalization)
        train_dataset = tf.keras.utils.timeseries_dataset_from_array(
            currency_data_temp,
            targets=currency_data_temp[delay:],
            sampling_rate=sampling_rate,
            sequence_length=sequence_length,
            batch_size=currencies_raw_data[currency][f"usd_{currency}_train_samples_number"],
        )

        # train dataset of evals of previous models
        train_dataset_for_eval_prev_models = tf.keras.utils.timeseries_dataset_from_array(
            currencies_raw_data[currency][f"usd_{currency}_raw_data_for_evals"],
            targets=currencies_raw_data[currency][f"usd_{currency}_raw_data_for_evals"][delay:],
            sampling_rate=sampling_rate,
            sequence_length=sequence_length,
            batch_size=currencies_raw_data[currency][f"usd_{currency}_train_samples_number"],
        )

        if index == 0:
            train_datasets[f"train_dataset_usd_{currency}_delay_5"] = train_dataset
            train_datasets_for_eval_prev_models[f"train_dataset_usd_{currency}_delay_5"] = train_dataset_for_eval_prev_models
        if index == 1:
            train_datasets[f"train_dataset_usd_{currency}_delay_25"] = train_dataset
            train_datasets_for_eval_prev_models[f"train_dataset_usd_{currency}_delay_25"] = train_dataset_for_eval_prev_models
        if index == 2:
            train_datasets[f"train_dataset_usd_{currency}_delay_45"] = train_dataset
            train_datasets_for_eval_prev_models[f"train_dataset_usd_{currency}_delay_45"] = train_dataset_for_eval_prev_models

        # Extracting data inputs and outputs
        # Based on the code provided in Chapter 10 (Deep Learning For Timeseries) of the book Deep Learning with Python by Francois Chollet [1].
        
        for samples, targets in train_dataset:
            # print("Samples: ", samples)
            # print("Sample shape: ", samples.shape)
            # print("Targets: ", targets)
            # print("Targets shape: ", targets.shape)
            data_inputs = tf.make_ndarray(tf.make_tensor_proto(samples))
            data_outputs = tf.make_ndarray(tf.make_tensor_proto(targets))

            if index == 0:
                data_inputs_test_all[f"{currency}_delay_5"] = data_inputs[-200:]
                data_outputs_test_all[f"{currency}_delay_5"] = data_outputs[-200:]
                data_inputs_all[f"{currency}_delay_5"] = data_inputs[:-200]
                data_outputs_all[f"{currency}_delay_5"] = data_outputs[:-200]
            if index == 1:
                data_inputs_test_all[f"{currency}_delay_25"] = data_inputs[-200:]
                data_outputs_test_all[f"{currency}_delay_25"] = data_outputs[-200:]
                data_inputs_all[f"{currency}_delay_25"] = data_inputs[:-200]
                data_outputs_all[f"{currency}_delay_25"] = data_outputs[:-200]
            if index == 2:
                data_inputs_test_all[f"{currency}_delay_45"] = data_inputs[-200:]
                data_outputs_test_all[f"{currency}_delay_45"] = data_outputs[-200:]
                data_inputs_all[f"{currency}_delay_45"] = data_inputs[:-200]
                data_outputs_all[f"{currency}_delay_45"] = data_outputs[:-200]

            # print(data_inputs)

        # Extracting data inputs and outputs for dataset for evals of prev models
        
        for samples, targets in train_dataset_for_eval_prev_models:
            # print("Samples: ", samples)
            # print("Sample shape: ", samples.shape)
            # print("Targets: ", targets)
            # print("Targets shape: ", targets.shape)
            data_inputs = tf.make_ndarray(tf.make_tensor_proto(samples))
            data_outputs = tf.reshape(tf.make_ndarray(tf.make_tensor_proto(targets)), [-1,1])

            if index == 0:
                data_inputs_test_all_for_eval_prev_models[f"{currency}_delay_5"] = data_inputs[-200:]
                data_outputs_test_all_for_eval_prev_models[f"{currency}_delay_5"] = data_outputs[-200:]
            if index == 1:
                data_inputs_test_all_for_eval_prev_models[f"{currency}_delay_25"] = data_inputs[-200:]
                data_outputs_test_all_for_eval_prev_models[f"{currency}_delay_25"] = data_outputs[-200:]
            if index == 2:
                data_inputs_test_all_for_eval_prev_models[f"{currency}_delay_45"] = data_inputs[-200:]
                data_outputs_test_all_for_eval_prev_models[f"{currency}_delay_45"] = data_outputs[-200:]

2026-02-22 13:50:26.042779: I tensorflow/core/framework/local_rendezvous.cc:407] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


### - Kerastuner implementation to obtain best model stucture

The code is based on the code provided in the Keras documentation for the KerasTuner [2].

Find the best hyperparameters for this new type of challenge.

In [21]:
from keras import models
from keras import layers
import keras_tuner

# Build model function
def build_lstm_model_iteration_two(hp):
    model = models.Sequential()
    model.add(
        layers.LSTM(
            units = hp.Int("units", min_value=16, max_value=64, step=4),
            input_shape=(sequence_length, 2)
        )
    )
    
    for i in range(2):
        model.add(
            layers.Dense(
                units = hp.Int(f"units_{i}", min_value=4, max_value=56, step=4),
                activation = hp.Choice("activation", ["relu", "linear"])
            )
        )
    
    model.add(
        layers.Dense(
            units = 1
        )
    )
    model.compile(optimizer="rmsprop", loss="mse", metrics=["mae"])
    return model

# 

tuner = keras_tuner.RandomSearch(
    hypermodel=build_lstm_model_iteration_two,
    objective="val_mae",
    max_trials=10,
    executions_per_trial=5,
    overwrite=True,
    directory="tuner_results/final_project_ml_project_iteration_two_tuner_best_model_results",
    project_name="final_project_ml_project_iteration_two_tuner_best_model"
)

tuner.search_space_summary()

Search space summary
Default search space size: 4
units (Int)
{'default': None, 'conditions': [], 'min_value': 16, 'max_value': 64, 'step': 4, 'sampling': 'linear'}
units_0 (Int)
{'default': None, 'conditions': [], 'min_value': 4, 'max_value': 56, 'step': 4, 'sampling': 'linear'}
activation (Choice)
{'default': 'relu', 'conditions': [], 'values': ['relu', 'linear'], 'ordered': False}
units_1 (Int)
{'default': None, 'conditions': [], 'min_value': 4, 'max_value': 56, 'step': 4, 'sampling': 'linear'}


/Users/studentcode/Documents/UOL/Semester 6/Computer Science Final Project/ML - CM3070 - In Final Project report - University of London/.venv/lib/python3.13/site-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Run the tuner using the data of the USD/JPY currency pair (as the objective for this step is to get a model structure)

In [22]:
import os

# check if ml_process_tuner_best_model exists
is_ml_process_tuner_best_model_created = os.path.exists("models/ml_project_iteration_two_tuner_best_model.keras")

# if it does not exists run the tuner
if is_ml_process_tuner_best_model_created != True:
    tuner.search(data_inputs_all["jpy_delay_5"], data_outputs_all["jpy_delay_5"], epochs=20, validation_data=(data_inputs_test_all["jpy_delay_5"], data_outputs_test_all["jpy_delay_5"]))
    best_model = tuner.get_best_models(num_models=1)[0]
    best_model.summary()

    # save model
    best_model.save("models/ml_project_iteration_two_tuner_best_model.keras")

Trial 10 Complete [00h 05m 00s]
val_mae: 0.13604446351528168

Best val_mae So Far: 0.1357489675283432
Total elapsed time: 00h 52m 11s


/Users/studentcode/Documents/UOL/Semester 6/Computer Science Final Project/ML - CM3070 - In Final Project report - University of London/.venv/lib/python3.13/site-packages/keras/src/saving/saving_lib.py:797: UserWarning: Skipping variable loading for optimizer 'rmsprop', because it has 2 variables whereas the saved optimizer has 11 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm (LSTM)                     │ (None, 36)             │         5,616 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 52)             │         1,924 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 16)             │           848 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 1)              │            17 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 8,405 (32.83 KB)

 Trainable params: 8,405 (32.83 KB)

 Non-trainable params: 0 (0.00 B)

Get the model layers

In [23]:
from keras.models import load_model

loaded_model = load_model("models/ml_project_iteration_two_tuner_best_model.keras")

# Get which activation functions were used
for layer in loaded_model.layers:
    print(layer.activation)

<function tanh at 0x138dd7380>
<function relu at 0x13819bf60>
<function relu at 0x13819bf60>
<function linear at 0x138dd7a60>


/Users/studentcode/Documents/UOL/Semester 6/Computer Science Final Project/ML - CM3070 - In Final Project report - University of London/.venv/lib/python3.13/site-packages/keras/src/saving/saving_lib.py:797: UserWarning: Skipping variable loading for optimizer 'rmsprop', because it has 11 variables whereas the saved optimizer has 2 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


### - Kerastuner implementation to train all models

In [27]:
from keras import layers
from keras import activations

def build_lstm_model_iteration_two_custom(hp):
    model = models.Sequential()
    model.add(layers.LSTM(36, input_shape=(sequence_length, 2)))
    model.add(layers.Dense(52, activation=activations.relu))
    model.add(layers.Dense(16, activation=activations.relu))
    model.add(layers.Dense(1, activation=activations.linear))
    model.compile(optimizer="rmsprop", loss="mse", metrics=["mae"])
    model.summary()
    return model

In [28]:
import keras_tuner
import os

# using KerasTuner to get three models per currency
for index, delay in enumerate(delays):
    real_delay = 0
    if index == 0:
        real_delay = 5
    if index == 1:
        real_delay = 25
    if index == 2:
        real_delay = 45
    print(f"---- ---- ---- ---- ---- ---- ---- ----\nDelay: {real_delay} days.")
    for currency in currencies:
        print(f"---- ---- ---- ----\nCurrency: {currency}.")

        tuner = keras_tuner.RandomSearch(
            hypermodel=build_lstm_model_iteration_two_custom,
            objective="val_mae",
            max_trials=1,
            executions_per_trial=5,
            overwrite=True,
            directory="tuner_results/final_project_ml_project_iteration_two_tuner_results",
            project_name="final_project_ml_project_iteration_two_tuner"
        )

        # check if ml_process_tuner_best_model exists
        is_ml_process_tuner_best_model_created = os.path.exists(f"models/ml_project_iteration_two_best_model_{currency}_delay_{real_delay}.keras")

        # if it does not exists run the tuner
        if is_ml_process_tuner_best_model_created != True:
            tuner.search(data_inputs_all[f"{currency}_delay_{real_delay}"], data_outputs_all[f"{currency}_delay_{real_delay}"], epochs=20, validation_data=(data_inputs_test_all[f"{currency}_delay_{real_delay}"], data_outputs_test_all[f"{currency}_delay_{real_delay}"]))
            best_model = tuner.get_best_models(num_models=1)[0]
            best_model.summary()

            # save model
            best_model.save(f"models/ml_project_iteration_two_best_model_{currency}_delay_{real_delay}.keras")

Trial 1 Complete [00h 05m 01s]
val_mae: 0.7192398905754089

Best val_mae So Far: 0.7192398905754089
Total elapsed time: 00h 05m 01s


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm (LSTM)                     │ (None, 36)             │         5,616 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 52)             │         1,924 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 16)             │           848 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 1)              │            17 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 8,405 (32.83 KB)

 Trainable params: 8,405 (32.83 KB)

...

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm (LSTM)                     │ (None, 36)             │         5,616 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 52)             │         1,924 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 16)             │           848 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 1)              │            17 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 8,405 (32.83 KB)

 Trainable params: 8,405 (32.83 KB)

 Non-trainable params: 0 (0.00 B)

### - Resulting models evaluation

In [50]:
from tabulate import tabulate

# Evaluation and comparison between the MAE of the GA models, the MAE of ML Project Iteration (KerasTuner) models,
# and the MAE of ML Project Iteration Two models
delay_five_table = []
delay_twentyfive_table = []
delay_fortyfive_table = []

for index, delay in enumerate(delays):
    real_delay = 0
    if index == 0:
        real_delay = 5
    if index == 1:
        real_delay = 25
    if index == 2:
        real_delay = 45
    # print(f"---- ---- ---- ---- ---- ---- ---- ----\nDelay: {real_delay} days.")
    for currency in currencies:
        print(f"---- ---- ---- ----\nCurrency: {currency}.")
        ga_model_loaded = tf.keras.models.load_model(f"models/model_{currency}_delay_{real_delay}.keras")
        iteration_one_model_loaded = tf.keras.models.load_model(f"models/ml_project_iteration_best_model_{currency}_delay_{real_delay}.keras")
        iteration_two_model_loaded = tf.keras.models.load_model(f"models/ml_project_iteration_two_best_model_{currency}_delay_{real_delay}.keras")
        test_mae_ga_model_loaded = ga_model_loaded.evaluate(data_inputs_test_all_for_eval_prev_models[f"{currency}_delay_{real_delay}"], data_outputs_test_all_for_eval_prev_models[f"{currency}_delay_{real_delay}"])[1]
        test_mae_iteration_one_model_loaded = iteration_one_model_loaded.evaluate(data_inputs_test_all_for_eval_prev_models[f"{currency}_delay_{real_delay}"], data_outputs_test_all_for_eval_prev_models[f"{currency}_delay_{real_delay}"])[1]
        test_mae_iteration_two_model_loaded = iteration_two_model_loaded.evaluate(data_inputs_test_all[f"{currency}_delay_{real_delay}"], data_outputs_test_all[f"{currency}_delay_{real_delay}"])[1]
        if real_delay == 5:
            delay_five_table.append([f"USD/{currency.upper()}", test_mae_ga_model_loaded, test_mae_iteration_one_model_loaded, test_mae_iteration_two_model_loaded])
        if real_delay == 25:
            delay_twentyfive_table.append([f"USD/{currency.upper()}",test_mae_ga_model_loaded, test_mae_iteration_one_model_loaded, test_mae_iteration_two_model_loaded])
        if real_delay == 45:
            delay_fortyfive_table.append([f"USD/{currency.upper()}",test_mae_ga_model_loaded, test_mae_iteration_one_model_loaded, test_mae_iteration_two_model_loaded])


---- ---- ---- ----
Currency: jpy.


/Users/studentcode/Documents/UOL/Semester 6/Computer Science Final Project/ML - CM3070 - In Final Project report - University of London/.venv/lib/python3.13/site-packages/keras/src/saving/saving_lib.py:797: UserWarning: Skipping variable loading for optimizer 'rmsprop', because it has 11 variables whereas the saved optimizer has 2 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - loss: 371.2234 - mae: 18.7104 
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: 5.3318 - mae: 1.9388  
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 0.0507 - mae: 0.1353  
---- ---- ---- ----
Currency: aud.
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - loss: 0.2192 - mae: 0.4663 
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - loss: 1.7801e-04 - mae: 0.0106 
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 0.3530 - mae: 0.5876  
---- ---- ---- ----
Currency: cad.
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: 0.0023 - mae: 0.0446  
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - loss: 8.4491e-05 - mae: 0.0075 
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 0.4470 - mae: 0.6653  
---- ---- ---- ----
Currency: cny.
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - loss: 0.0021 - mae: 0.0392 
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - loss: 4.0277e-04 - mae: 0.0155 
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 1.0643 - mae: 1.0231  
---- ---- ---- ----
Currency: hkd.
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 9

Create the tables to compare the results

In [51]:
# Create the table for the 5 day delay
print(tabulate(delay_five_table, headers=["Currency", "MAE - GA", "MAE - ML Project Iteration 1",  "MAE - ML Project Iteration 2"], tablefmt="grid"))

+------------+------------+--------------------------------+--------------------------------+
| Currency   |   MAE - GA |   MAE - ML Project Iteration 1 |   MAE - ML Project Iteration 2 |
+============+============+================================+================================+
| USD/JPY    | 18.7104    |                     1.93882    |                       0.135255 |
+------------+------------+--------------------------------+--------------------------------+
| USD/AUD    |  0.466277  |                     0.0105518  |                       0.587577 |
+------------+------------+--------------------------------+--------------------------------+
| USD/CAD    |  0.0446317 |                     0.00746274 |                       0.665284 |
+------------+------------+--------------------------------+--------------------------------+
| USD/CNY    |  0.0391992 |                     0.0155393  |                       1.02308  |
+------------+------------+--------------------------------+

In [52]:
# Create the table for the 25 day delay
print(tabulate(delay_twentyfive_table, headers=["Currency", "MAE - GA", "MAE - ML Project Iteration 1",  "MAE - ML Project Iteration 2"], tablefmt="grid"))

+------------+------------+--------------------------------+--------------------------------+
| Currency   |   MAE - GA |   MAE - ML Project Iteration 1 |   MAE - ML Project Iteration 2 |
+============+============+================================+================================+
| USD/JPY    | 28.249     |                      2.83452   |                       0.136543 |
+------------+------------+--------------------------------+--------------------------------+
| USD/AUD    |  1.13943   |                      0.0155922 |                       0.587577 |
+------------+------------+--------------------------------+--------------------------------+
| USD/CAD    |  0.0404597 |                      0.0157466 |                       0.665284 |
+------------+------------+--------------------------------+--------------------------------+
| USD/CNY    |  0.605407  |                      0.037116  |                       1.02308  |
+------------+------------+--------------------------------+

In [53]:
# Create the table for the 45 day delay
print(tabulate(delay_fortyfive_table, headers=["Currency", "MAE - GA", "MAE - ML Project Iteration 1",  "MAE - ML Project Iteration 2"], tablefmt="grid"))

+------------+------------+--------------------------------+--------------------------------+
| Currency   |   MAE - GA |   MAE - ML Project Iteration 1 |   MAE - ML Project Iteration 2 |
+============+============+================================+================================+
| USD/JPY    | 23.3651    |                      3.52465   |                       0.134377 |
+------------+------------+--------------------------------+--------------------------------+
| USD/AUD    |  0.0968492 |                      0.0156326 |                       0.587577 |
+------------+------------+--------------------------------+--------------------------------+
| USD/CAD    |  0.232504  |                      0.0191894 |                       0.665284 |
+------------+------------+--------------------------------+--------------------------------+
| USD/CNY    |  0.289527  |                      0.0322847 |                       1.02308  |
+------------+------------+--------------------------------+

## About the code

The timeseries code is based on the code provided in Chapter 10 (Deep Learning For Timeseries) of the book Deep Learning with Python by Francois Chollet [1].

The KerasTuner code is based on the code provided in the Keras documentation for the KerasTuner [2].

## References

1- Francois Chollet. 2021. Deep Learning with Python, Second Edition. Chapter 10, Deep learning for timeseries, Preparing the data. Retrieved from https://learning.oreilly.com/library/view/deep-learning-with/9781617296864/Text/10.htm#heading_id_5

2- Keras. Getting started with KerasTuner. Retrieved from https://keras.io/keras_tuner/getting_started/